# deepCab — Colab training + VS Code attach

Open this notebook on Colab (Runtime → Change runtime type → GPU). Run cells 1–3, then attach VS Code to the printed URL via *Cmd+Shift+P → Jupyter: Specify Jupyter Server for Connections*. From cell 4 onward, run cells from VS Code — code executes on Colab's GPU, edit happens locally.

**Colab secrets needed** (Tools → Secrets):
- `NGROK_AUTHTOKEN` — from https://dashboard.ngrok.com
- `GH_TOKEN` — fine-grained PAT with `Actions: write` for the deepCab repo
- `GCP_PROJECT` — your GCP project id
- `GH_REPO` — `<owner>/<name>` (e.g. `juan-garassino/deepCab`)

In [ ]:
# Cell 1 — setup
!pip install -q pyngrok jupyter_server
!git clone -q https://github.com/$GH_REPO_NAME.git /content/deepCab || (cd /content/deepCab && git pull -q)
%cd /content/deepCab/001-deepCab-api
!pip install -q -e .

In [ ]:
# Cell 2 — auth
from google.colab import auth, userdata
auth.authenticate_user()
NGROK_AUTHTOKEN = userdata.get('NGROK_AUTHTOKEN')
GH_TOKEN = userdata.get('GH_TOKEN')
GCP_PROJECT = userdata.get('GCP_PROJECT')
GH_REPO = userdata.get('GH_REPO')
assert all([NGROK_AUTHTOKEN, GH_TOKEN, GCP_PROJECT, GH_REPO]), 'set all 4 Colab secrets first'

In [ ]:
# Cell 3 — start jupyter server + ngrok tunnel
import secrets as _s, subprocess, time
from pyngrok import ngrok
TOKEN = _s.token_urlsafe(24)
ngrok.set_auth_token(NGROK_AUTHTOKEN)
tunnel = ngrok.connect(8888, 'http')
subprocess.Popen([
    'jupyter', 'server',
    '--ip=0.0.0.0', '--port=8888', '--no-browser',
    f'--ServerApp.token={TOKEN}',
    '--ServerApp.allow_origin=*',
    '--ServerApp.disable_check_xsrf=True',
])
time.sleep(4)
url = f'{tunnel.public_url}?token={TOKEN}'
print('VS Code → Jupyter: Specify Server →', url)
print('Then open this notebook locally and pick that server as the kernel.')

In [ ]:
# Cell 4 — train on Colab GPU (run from VS Code attached)
from deepCab.training.train import run
from deepCab.schemas.config import TrainConfig, TorchMLPConfig, DataRef
result = run(TrainConfig(
    backend=TorchMLPConfig(epochs=50, lr=1e-3, batch_size=512),
    data=DataRef(size='full'),
))
print(result.metrics)
RUN_ID = result.run_id

In [ ]:
# Cell 5 — push artifact to GCS
import subprocess
subprocess.run(['gsutil', '-m', 'cp', '-r', f'runs/{RUN_ID}', f'gs://deepcab-models/runs/{RUN_ID}/'], check=True)
MODEL_URI = f'gs://deepcab-models/runs/{RUN_ID}/'
print('uploaded:', MODEL_URI)

In [ ]:
# Cell 6 — trigger Cloud Run deploy
import requests
r = requests.post(
    f'https://api.github.com/repos/{GH_REPO}/actions/workflows/deploy-cloud-run.yml/dispatches',
    headers={'Authorization': f'token {GH_TOKEN}', 'Accept': 'application/vnd.github+json'},
    json={'ref': 'main', 'inputs': {'tag': RUN_ID, 'model_uri': MODEL_URI}},
)
r.raise_for_status()
print('deploy triggered for', MODEL_URI)